# Register NeMo TFM Pipelines

Run this notebook **once** from the workbench to register both pipelines with the
OpenShift AI pipeline server and optionally submit runs.

## Two pipelines — one notebook

| Pipeline | Name in dashboard | YAML | What it does |
|---|---|---|---|
| **Papermill** *(Section 4)* | `nemo-transaction-foundation-model` | `nemo_tfm_pipeline.yaml` | Each step runs a notebook via papermill. Produces executed `.ipynb` files with full cell-level output. Zero refactoring from the original notebooks. |
| **Decorated** *(Section 5)* | `nemo-tfm-foundation-model-decorated` | `foundation_model_pipeline_decorated.yaml` | True typed KFP components calling `src/` directly. Logs `auc_raw_features`, `auc_embeddings`, `lift_pct` as structured metrics to the KFP metadata store. Parameters recorded per-run. |

Sections 1–3 are shared setup (install SDK, locate YAMLs, connect).  
Section 4 registers the **papermill pipeline**.  
Section 5 registers the **decorated pipeline** (with the parameter reference table).

See `docs/notebooks-to-pipelines.md` for the full trade-off analysis.

---

**Pre-requisite:** clone the repo to `/opt/app-root/src/nemo-tfm` if this is a fresh workbench:
```bash
git clone https://github.com/robbybrodie/transaction-foundation-model-openshiftai.git \
  /opt/app-root/src/nemo-tfm
```

## 1 · Install KFP SDK

The NeMo image's venv is read-only, so we install `kfp` to a folder on the
shared PVC (`/opt/app-root/src/lib`).  This persists across pod restarts and
only needs to run once per workbench — skip this cell if `lib/` already exists.

In [ ]:
import subprocess, sys, os

LIB_DIR = "/opt/app-root/src/lib"

if not os.path.isdir(os.path.join(LIB_DIR, "kfp")):
    print("Installing kfp to PVC lib dir (one-time, ~60s)...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "--target", LIB_DIR,
        "--quiet",
        "kfp>=2.7,<3",
        "kfp-kubernetes>=1.3,<2",
    ])
    print("Done.")
else:
    print("kfp already installed in lib dir — skipping.")

# Add the PVC lib dir to the front of the path so it takes precedence
if LIB_DIR not in sys.path:
    sys.path.insert(0, LIB_DIR)

import kfp
print(f"kfp {kfp.__version__} ready")

## 2 · Locate the pipeline YAMLs

Both compiled YAMLs are committed to the repo — no recompilation needed.

In [ ]:
import os

# ---------------------------------------------------------------------------
# Decorated pipeline — DEFAULT (Sections 1–5)
# ---------------------------------------------------------------------------
DECORATED_NAME = "nemo-tfm-foundation-model-decorated"

# Locate repo root by finding the decorated pipeline YAML
for candidate in [
    "/opt/app-root/src/nemo-tfm",
    "/opt/app-root/src",
    "/opt/app-root/src/transaction-foundation-model-openshiftai",
]:
    _yaml = os.path.join(candidate, "pipeline", "foundation_model_pipeline_decorated.yaml")
    if os.path.isfile(_yaml):
        REPO_ROOT = candidate
        DECORATED_YAML = _yaml
        break
else:
    raise FileNotFoundError(
        "Cannot find pipeline/foundation_model_pipeline_decorated.yaml — "
        "clone the repo to /opt/app-root/src/nemo-tfm first:\n"
        "  git clone "
        "https://github.com/robbybrodie/transaction-foundation-model-openshiftai.git "
        "/opt/app-root/src/nemo-tfm"
    )

# ---------------------------------------------------------------------------
# Papermill pipeline — OPTIONAL (Section 6)
# ---------------------------------------------------------------------------
PAPERMILL_NAME = "nemo-transaction-foundation-model"
PAPERMILL_YAML = os.path.join(REPO_ROOT, "pipeline", "nemo_tfm_pipeline.yaml")

print(f"Repo root:       {REPO_ROOT}")
print(f"Decorated YAML:  {DECORATED_YAML}")
print(f"Papermill YAML:  {PAPERMILL_YAML}")

## 3 · Connect to the pipeline server

In [ ]:
import kfp

# In-cluster endpoint — port 8888 is the direct KFP API (HTTPS + SA token).
# Port 8443 is the OAuth proxy which rejects service account tokens; use 8888.
KFP_ENDPOINT = "https://ds-pipeline-pipelines-definition.nemo-tfm.svc.cluster.local:8888"
SA_TOKEN_PATH = "/var/run/secrets/kubernetes.io/serviceaccount/token"

with open(SA_TOKEN_PATH) as f:
    token = f.read().strip()

client = kfp.Client(
    host=KFP_ENDPOINT,
    existing_token=token,
    verify_ssl=False,   # cluster uses internal self-signed cert
)

# Smoke-test — list existing pipelines
existing = client.list_pipelines()
print(f"Connected. Pipelines already registered: {existing.total_size}")

## 4 · Register the papermill pipeline

The papermill pipeline (`nemo-transaction-foundation-model`) wraps each of the five
original notebooks in a `papermill` call. Every step produces an executed `.ipynb` file
with full cell-level output saved to `pipeline-outputs/` on the PVC.

**Use this pipeline when you want cell-by-cell notebook output for each step.**
It does not log structured metrics to the KFP metadata store.

In [ ]:
from datetime import datetime, timezone

# Version name — timestamp so the newest upload is always auto-selected in the UI
pm_version_name = f"v{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M')}"

existing = client.list_pipelines()
existing_ids = {
    p.display_name: p.pipeline_id
    for p in (existing.pipelines or [])
}

if PAPERMILL_NAME in existing_ids:
    pm_pipeline_id = existing_ids[PAPERMILL_NAME]
    pm_result = client.upload_pipeline_version(
        pipeline_package_path=PAPERMILL_YAML,
        pipeline_version_name=pm_version_name,
        pipeline_id=pm_pipeline_id,
    )
    print(f"Updated pipeline '{PAPERMILL_NAME}' — new version: {pm_version_name}")
    print(f"Version id: {pm_result.pipeline_version_id}")
else:
    pm_result = client.upload_pipeline(
        pipeline_package_path=PAPERMILL_YAML,
        pipeline_name=PAPERMILL_NAME,
    )
    print(f"Registered new pipeline '{PAPERMILL_NAME}'")
    print(f"Pipeline id: {pm_result.pipeline_id}")

print(f"\nDone. In the RHOAI dashboard:")
print(f"  Pipelines tab → {PAPERMILL_NAME} → Create run")
print(f"  Parameters: work_dir, mode (demo/train)")
print(f"\n  Executed notebooks saved to pipeline-outputs/ on the PVC after each run.")

### Papermill pipeline — steps

| Step | Notebook | What it does | demo | train |
|------|----------|--------------|:----:|:-----:|
| s1 | `01_dataset_baseline.ipynb` | TabFormer load, temporal splits, XGBoost baseline | ✓ | ✓ |
| s2 | `02_seq_preproc_tokenization.ipynb` | GPU-accelerated cuDF tokeniser; writes corpus | ✓ | ✓ |
| s3 | `03_foundation_model_training.ipynb` | NeMo decoder pretraining (30-step demo) | — | ✓ |
| s4 | `04_inference_embedding_extraction.ipynb` | Batch inference; extracts 512-d embeddings from LFS checkpoint | ✓ | ✓ |
| s5 | `05_xgboost_fraud_detection.ipynb` | XGBoost; prints AUC to notebook cell output | ✓ | ✓ |

**Parameters:** `work_dir` (repo root on PVC), `mode` (`"demo"` or `"train"`)

**Output:** Executed `.ipynb` files saved to `pipeline-outputs/` on the PVC.
No structured metrics are logged to the KFP metadata store.

### 4b · Submit a papermill run (optional)

Run the cell below to submit a papermill pipeline run from this notebook.
Executed notebooks are saved to `pipeline-outputs/` on the shared PVC.

In [ ]:
from datetime import datetime, timezone

PM_MODE = "demo"   # ← "demo" or "train"

pm_pipeline_id = existing_ids.get(PAPERMILL_NAME) or pm_result.pipeline_id

pm_run_name = (
    f"nemo-transaction-{PM_MODE}-"
    f"{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M')}"
)

pm_run = client.create_run_from_pipeline_package(
    pipeline_file=PAPERMILL_YAML,
    arguments={
        "work_dir": "/opt/app-root/src/nemo-tfm",
        "mode": PM_MODE,
    },
    run_name=pm_run_name,
    enable_caching=False,
)

print(f"Run submitted:  {pm_run_name}")
print(f"Pipeline:       {PAPERMILL_NAME}")
print(f"Mode:           {PM_MODE}")
print(f"Run ID:         {pm_run.run_id}")
print(f"\nWatch progress: RHOAI dashboard → Pipelines → Runs")
print(f"Cell outputs:   nemo-tfm-workbench:/opt/app-root/src/nemo-tfm/pipeline-outputs/")

---

## 5 · Register the decorated pipeline

The decorated pipeline (`nemo-tfm-foundation-model-decorated`) uses true typed KFP
components that call `src/` directly. It logs `auc_raw_features`, `auc_embeddings`,
and `lift_pct` as structured metrics to the KFP metadata store.

**Use this pipeline when you want dashboard metrics queryable across runs, or when
preparing governance evidence (APRA CPG 220 audit trail).**

See `docs/notebooks-to-pipelines.md` for the full trade-off analysis.

In [ ]:
from datetime import datetime, timezone

# Version name — timestamp so the newest upload is always auto-selected in the UI
version_name = f"v{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M')}"

# Re-fetch current pipeline list (in case Section 4 was run in a previous session)
existing = client.list_pipelines()
existing_ids_all = {
    p.display_name: p.pipeline_id
    for p in (existing.pipelines or [])
}

if DECORATED_NAME in existing_ids_all:
    pipeline_id = existing_ids_all[DECORATED_NAME]
    result = client.upload_pipeline_version(
        pipeline_package_path=DECORATED_YAML,
        pipeline_version_name=version_name,
        pipeline_id=pipeline_id,
    )
    print(f"Updated pipeline '{DECORATED_NAME}' — new version: {version_name}")
    print(f"Version id: {result.pipeline_version_id}")
else:
    result = client.upload_pipeline(
        pipeline_package_path=DECORATED_YAML,
        pipeline_name=DECORATED_NAME,
    )
    print(f"Registered new pipeline '{DECORATED_NAME}'")
    print(f"Pipeline id: {result.pipeline_id}")

print(f"\nDone. In the RHOAI dashboard:")
print(f"  Pipelines tab → {DECORATED_NAME} → Create run")
print(f"  The version '{version_name}' will be pre-selected (most recent).")
print(f"\n  Set 'mode' to 'demo' (default, ~20-30 min) or 'train' (full run).")
print(f"  Metrics (auc_raw_features, auc_embeddings, lift_pct) appear in the Runs → Metrics tab.")

### Decorated pipeline — steps, parameters, and metrics

#### Steps

| Step | Component | What it does | demo | train |
|------|-----------|--------------|:----:|:-----:|
| s0 | `prepare_dataset` | Downloads TabFormer CSV, creates 80/10/10 temporal splits + 100K eval subsets; CPU only | ✓ | ✓ |
| s1 | `tokenize_transactions` | GPU-accelerated cuDF tokeniser; writes corpus to `data/decoder_corpus/` | ✓ | ✓ |
| s2 | `train_foundation_model` | NeMo decoder pretraining via `torchrun`; records `max_steps` in metadata store | — | ✓ |
| s3 | `extract_embeddings` | Batch inference; extracts 512-d last-token embeddings from LFS checkpoint | ✓ | ✓ |
| s4 | `evaluate_fraud_detection` | PCA 64d + XGBoost (3 models); logs 5 structured metrics | ✓ | ✓ |

**Demo mode** (default): s0 → s1 → s3 → s4  (4 steps, ~20–30 min on L4)  
**Train mode:** s0 → s1 → s2 → s3 → s4  (5 steps)

#### Pipeline parameters

| Parameter | Default | Notes |
|-----------|---------|-------|
| `mode` | `"demo"` | `"demo"` skips training (~20-30 min). `"train"` runs all steps. |
| `work_dir` | `/opt/app-root/src/nemo-tfm` | Repo root on the shared PVC. |
| `model_path` | `models/decoder-foundation-model` | Path to checkpoint for embedding extraction. Relative paths resolved from `work_dir`. |
| `max_steps` | `30` | Training steps (train mode only). `30` = demo; increase for real training. |
| `force_rerun` | `False` | `False`: reuse existing splits/corpus/embeddings for fast repeated runs. `True`: clear and regenerate all intermediate outputs (always use `True` for train mode). |

#### Metrics (visible in dashboard Runs → Metrics tab)

| Metric | Value | Description |
|--------|-------|-------------|
| `auc_raw_features` | ~0.9629 | Test ROC-AUC for XGBoost trained on raw tabular features only (13d) |
| `auc_embeddings` | ~0.8742 | Test ROC-AUC for XGBoost trained on 64-d PCA of foundation model embeddings alone |
| `lift_pct` | ~−9.21% | Embeddings-only lift over baseline — **expected to be negative**. Embeddings alone do not beat well-engineered tabular features. Recording this is honest and makes the combined result credible. |
| `auc_combined` | ~0.9803 | Test ROC-AUC for XGBoost trained on raw (13d) + PCA embeddings (64d) = 77d |
| `lift_combined_pct` | ~+1.82% | **Headline number.** Combined model lift over raw baseline. Decoder embeddings carry signal that raw features do not have. |

**Honest reading:** `lift_pct` is negative by design — the embeddings are not a substitute for
well-chosen tabular features, they are a complement. The `lift_combined_pct` is the meaningful
result: adding embeddings to the raw features improves AUC by ~+1.82% over the already-strong
baseline. Both numbers are recorded so any governance reviewer can see the full picture, not just
the positive result.

**Note:** In both modes, `extract_embeddings` uses `models/decoder-foundation-model/`
(the Git LFS 3000-step NVIDIA checkpoint). The proven checkpoint ensures the accuracy
story is consistent across all runs.

### 5b · Submit a decorated run (optional)

The decorated pipeline exposes a `mode` parameter and a `force_rerun` flag.
Run the cell below to submit directly from this notebook, or use the RHOAI dashboard
under **Pipelines → Create run → Parameters**.

Five metrics appear in **Runs → Metrics tab** — queryable across all runs without opening a notebook:

| Metric | Description |
|--------|-------------|
| `auc_raw_features` | ~0.9629 — raw 13d feature baseline |
| `auc_embeddings` | ~0.8742 — embeddings-only (PCA 64d); expected to be lower than baseline |
| `lift_pct` | ~−9.21% — embeddings-only lift; negative is expected and honest |
| `auc_combined` | ~0.9803 — raw + embeddings combined (77d) |
| `lift_combined_pct` | ~+1.82% — **headline number**; combined model lift over raw baseline |

In [ ]:
from datetime import datetime, timezone

# ── Choose mode ────────────────────────────────────────────────────────────
#
#   "demo"  — tokenise → extract embeddings → evaluate  (~20-30 min on L4)
#             Skips training. Uses the Git LFS 3000-step checkpoint for
#             embedding extraction. Accuracy story is reproducible and fast.
#             Use this for every demo.
#
#   "train" — tokenise → train → extract embeddings → evaluate
#             Runs a 30-step demo training to show the capability. Embedding
#             extraction still uses the Git LFS checkpoint, not the 30-step output.
#             Set force_rerun=True to avoid stale demo artifacts.
#
MODE = "demo"   # ← change to "train" to include the training step

dec_pipeline_id = existing_ids_all.get(DECORATED_NAME) or result.pipeline_id

run_name = (
    f"nemo-tfm-{MODE}-"
    f"{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M')}"
)

run = client.create_run_from_pipeline_package(
    pipeline_file=DECORATED_YAML,
    arguments={
        "work_dir": "/opt/app-root/src/nemo-tfm",
        "mode": MODE,
        # "force_rerun": True,   # ← uncomment to clear stale artifacts
    },
    run_name=run_name,
    enable_caching=False,
)

print(f"Run submitted:  {run_name}")
print(f"Pipeline:       {DECORATED_NAME}")
print(f"Mode:           {MODE}")
print(f"Run ID:         {run.run_id}")
print(f"\nWatch progress: RHOAI dashboard → Pipelines → Runs")
print(f"View metrics:   Runs → {run_name} → Metrics tab")